In [ ]:
# Uncomment if needed
# !pip install torch torchvision Pillow numpy matplotlib

In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as transforms

from google.colab import drive

import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

In [ ]:
# Auto-detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = "https://github.com/your-username/your-repo.git"
    REPO_DIR = "/content/your-repo"
    
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}

    !cp -r "" /content/
    
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
else:
    src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
    sys.path.insert(0, src_path)

In [ ]:
from autoencoder import *

In [ ]:
# load data
data_file = ""
data = np.load(data_file)

train_tensor = torch.from_numpy(data['train'].astype(np.float32))
train_tensor = torch.from_numpy(data['test'].astype(np.float32))

print(f'Training Data Size: {train_data.shape}')
print(f'Test Data Size: {test_data.shape}')

train_tensor_dataset = TensorDataset(train_tensor, torch.zeros(len(all_images), dtype=torch.long))
test_tensor_dataset = TensorDataset(test_tensor, torch.zeros(len(all_images), dtype=torch.long))

train_loader = DataLoader(train_tensor_dataset, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_tensor_dataset,  batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# display image
xb, _ = next(iter(train_loader))
print("Batch shape:", xb.shape)

plt.figure(figsize=(4,4))

if xb.shape[1] == 1:
    plt.imshow(xb[0].squeeze(), cmap="gray")
    plt.title("Example grayscale input")
else:
    plt.imshow(xb[0].permute(1, 2, 0))
    plt.title("Example color input")

plt.axis("off")
plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def train_autoencoder_sharp(
    autoencoder, discriminator, train_loader, test_loader=None,
    epochs=150, lr_ae=1e-4, lr_disc=4e-4,
    max_kl_weight=1e-6, kl_warmup_epochs=30,
    disc_start_epoch=30, disc_weight=0.1,
):
    device = next(autoencoder.parameters()).device
    recon_criterion = SharpReconLoss().to(device)

    opt_ae = torch.optim.AdamW(
        autoencoder.parameters(), lr=lr_ae, betas=(0.5, 0.9),
        weight_decay=1e-4
    )
    opt_disc = torch.optim.AdamW(
        discriminator.parameters(), lr=lr_disc, betas=(0.5, 0.9),
        weight_decay=1e-4
    )

    sched_ae = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ae, epochs)
    sched_disc = torch.optim.lr_scheduler.CosineAnnealingLR(opt_disc, epochs)

    for epoch in range(epochs):
        autoencoder.train()
        discriminator.train()
        use_disc = epoch >= disc_start_epoch

        # KL warmup
        if epoch < kl_warmup_epochs:
            kl_weight = max_kl_weight * (epoch / kl_warmup_epochs)
        else:
            kl_weight = max_kl_weight

        metrics = {'recon': 0, 'kl': 0, 'g_loss': 0, 'd_loss': 0}

        for images, _ in train_loader:
            images = images.to(device)
            recon, mean, logvar, z = autoencoder(images)

            # ── Train Discriminator ──
            if use_disc:
                # Real
                real_pred = discriminator(images)
                d_real = F.relu(1.0 - real_pred).mean()

                # Fake
                fake_pred = discriminator(recon.detach())
                d_fake = F.relu(1.0 + fake_pred).mean()

                d_loss = (d_real + d_fake) * 0.5

                opt_disc.zero_grad()
                d_loss.backward()
                opt_disc.step()
                metrics['d_loss'] += d_loss.item()

            # ── Train Autoencoder ──
            recon_loss, loss_dict = recon_criterion(recon, images)
            kl_loss = -0.5 * torch.mean(
                1 + logvar - mean.pow(2) - logvar.exp()
            )

            ae_loss = recon_loss + kl_weight * kl_loss

            # Generator loss — fool discriminator
            if use_disc:
                fake_pred = discriminator(recon)
                g_loss = -fake_pred.mean()
                ae_loss = ae_loss + disc_weight * g_loss
                metrics['g_loss'] += g_loss.item()

            opt_ae.zero_grad()
            ae_loss.backward()
            torch.nn.utils.clip_grad_norm_(autoencoder.parameters(), 1.0)
            opt_ae.step()

            metrics['recon'] += loss_dict['l1']
            metrics['kl'] += kl_loss.item()

        sched_ae.step()
        sched_disc.step()
        n = len(train_loader)

        # Validation
        val_str = ""
        if test_loader is not None:
            autoencoder.eval()
            val_loss = 0
            with torch.no_grad():
                for images, _ in test_loader:
                    images = images.to(device)
                    recon, _, _, _ = autoencoder(images)
                    val_loss += F.l1_loss(recon, images).item()
            val_str = f" | Val: {val_loss/len(test_loader):.4f}"

        disc_str = ""
        if use_disc:
            disc_str = (f" | D: {metrics['d_loss']/n:.4f}"
                        f" | G: {metrics['g_loss']/n:.4f}")

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Recon: {metrics['recon']/n:.4f} | "
              f"KL: {metrics['kl']/n:.4f} | "
              f"KL_w: {kl_weight:.7f}"
              f"{disc_str}{val_str}")

autoencoder = SpatialVAE(in_ch=1, latent_ch=3).to(device)
discriminator = PatchDiscriminator(in_ch=1).to(device)

train_autoencoder_sharp(
    autoencoder,
    discriminator,
    train_loader,
    test_loader=test_loader,
    epochs=60,
    max_kl_weight=1e-6,       # very small — prioritize reconstruction
    kl_warmup_epochs=30,       # no KL for first 30 epochs
    disc_start_epoch=30,       # no GAN for first 30 epochs
    disc_weight=0.1,           # gentle adversarial pressure
)

In [ ]:
torch.save(autoencoder, '/content/drive/MyDrive/ENGS 106/models/spatial_autoencoder.pth')